# Batch Box Segmentation with SAM2

**5-stage pipeline:**
1. SAM2 AutoMaskGenerator → candidate masks
2. Geometric filter (rectangularity + fully-in-frame)
3. Texture filter — edge density + local variance *(toggle `ENABLE_TEXTURE_FILTER`)*
4. SAM2 Predictor bbox-prompt refinement
5. Morphological clean-up → save RGBA per box

**Key parameters:**
- Set `ENABLE_TEXTURE_FILTER = False` for the original 4-stage version
- Set `MAX_IMAGE_SIZE = 1024` to prevent CUDA OOM on large images
- Texture filter is CPU-side (skimage/scipy), zero GPU overhead

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc
import glob
import time
import numpy as np
import torch
from PIL import Image
from skimage import morphology
from skimage.morphology import opening, closing, disk
from skimage.color import rgb2gray
from skimage.filters import sobel
from scipy.ndimage import uniform_filter

from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
from sam2.sam2_image_predictor import SAM2ImagePredictor

print("Imports ready.")

## 1. Configuration & Hyperparameters

Tune these knobs based on your box size, image resolution, and speed/accuracy trade-off.

In [ ]:
# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
INPUT_DIR  = "/workspace/sam/input"
OUTPUT_DIR = "/workspace/sam/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# SAM2 checkpoint and config
CHECKPOINT = "/workspace/segment-anything-2/checkpoints/sam2.1_hiera_large.pt"
CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ------------------------------------------------------------------
# Image resizing (prevents OOM on huge images)
# ------------------------------------------------------------------
MAX_IMAGE_SIZE = 1024      # Resize so the longest side is this many pixels
                           # Increase to 1536 if you have >16GB GPU

# ------------------------------------------------------------------
# Stage 1 — Automatic Mask Generator (tuned for speed + memory)
# ------------------------------------------------------------------
POINTS_PER_SIDE = 12        # Reduced from 20 → saves ~50% GPU memory
PRED_IOU_THRESH = 0.85
STABILITY_SCORE_THRESH = 0.90
CROP_N_LAYERS = 1
CROP_N_POINTS_DOWNSCALE_FACTOR = 2
MIN_MASK_REGION_AREA = 200

# ------------------------------------------------------------------
# Stage 2 — Geometric Filtering
# ------------------------------------------------------------------
MIN_RECTANGULARITY = 0.45
MAX_ASPECT_RATIO = 6.0
MIN_AREA_RATIO = 0.005
MAX_AREA_RATIO = 0.95

BOUNDARY_MARGIN = 8

# ------------------------------------------------------------------
# Stage 2.5 — Texture Feature Filtering
# ------------------------------------------------------------------
# Rationale: brick walls & repetitive patterns pass Stage 2
# (rectangular shape) but have different surface statistics.
#
# Disable with ENABLE_TEXTURE_FILTER=False for the original 4-stage pipeline.
#
# Texture features computed per mask:
#   edge_density  = mean Sobel gradient magnitude (real boxes have
#                   labels/tape/creases → more edges)
#   local_std     = mean of local 7×7 patch std (boxes have
#                   non-uniform surfaces vs flat brick faces)
#   std_of_std    = std of local std values (brick walls are
#                   bimodal: flat bricks + sharp mortar edges)
# ------------------------------------------------------------------
ENABLE_TEXTURE_FILTER = True

# Real boxes: edge_density >= this threshold
# Lower → keep more; raise if false positives persist
TEXTURE_MIN_EDGE_DENSITY = 0.04

# Real boxes: mean local 7×7 std >= this threshold
# Brick walls have flat brick faces → low local_std
TEXTURE_MIN_LOCAL_STD = 0.015

# Brick walls have bimodal local-std distribution
# (most patches = smooth brick, few = sharp mortar)
# → std_of_std is high. Real boxes are more uniform.
# Set 0 to disable this check.
TEXTURE_MAX_STD_OF_STD = 0.03

# ------------------------------------------------------------------
# Stage 3 — Predictor Refinement
# ------------------------------------------------------------------
REFINE_WITH_PREDICTOR = True
REFINE_MIN_SCORE = 0.80

# ------------------------------------------------------------------
# Stage 4 — Post-processing
# ------------------------------------------------------------------
MORPH_OPEN_RADIUS = 2
MORPH_CLOSE_RADIUS = 3

print("Configuration loaded.")

## 2. Helper Functions

In [ ]:
def get_mask_bbox(mask: np.ndarray):
    """Return (x1, y1, x2, y2) bounding box of a binary mask."""
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())


def is_fully_in_frame(bbox, img_h: int, img_w: int, margin: int = 8):
    """
    Return True only if the bbox is completely inside the image frame,
    with a safety margin from every edge.
    If the box touches or crosses the boundary → False (skip it).
    """
    x1, y1, x2, y2 = bbox
    return (x1 >= margin and y1 >= margin and
            x2 < img_w - margin and y2 < img_h - margin)


def is_box_like(mask: np.ndarray, img_h: int, img_w: int):
    """
    Check whether a mask looks like a box (rectangular, reasonable aspect ratio).
    Broken boxes are accepted because MIN_RECTANGULARITY is low.
    """
    # Keep only the largest connected component (ignore detached noise)
    labeled = morphology.label(mask, connectivity=2)
    if labeled.max() == 0:
        return False
    
    # Largest component
    component_sizes = np.bincount(labeled.ravel())
    component_sizes[0] = 0  # ignore background
    largest_label = component_sizes.argmax()
    clean_mask = (labeled == largest_label)
    
    area = clean_mask.sum()
    img_area = img_h * img_w
    area_ratio = area / img_area
    
    if area_ratio < MIN_AREA_RATIO or area_ratio > MAX_AREA_RATIO:
        return False
    
    ys, xs = np.where(clean_mask)
    x1, y1, x2, y2 = xs.min(), ys.min(), xs.max(), ys.max()
    w, h = x2 - x1 + 1, y2 - y1 + 1
    bbox_area = w * h
    
    # Aspect ratio check
    aspect = max(w, h) / (min(w, h) + 1e-6)
    if aspect > MAX_ASPECT_RATIO:
        return False
    
    # Rectangularity = mask area / bbox area (1.0 = perfect rectangle)
    rectangularity = area / (bbox_area + 1e-6)
    if rectangularity < MIN_RECTANGULARITY:
        return False
    
    return True


# ──────────────────────────────────────────────────────────────
# Stage 2.5 — Texture Feature Computation
# ──────────────────────────────────────────────────────────────

def compute_texture_features(mask: np.ndarray, image: np.ndarray):
    """
    Compute 3 texture features for the masked region.

    Features designed to separate real box surfaces (labels, tape,
    cardboard grain, creases) from brick walls (repetitive flat + edge).

    Returns:
        edge_density  — mean Sobel gradient magnitude in mask region
        local_std     — mean of 7x7 patch standard deviations
        std_of_std    — std of 7x7 patch standard deviations (bimodality)
    """
    gray = rgb2gray(image).astype(np.float64)

    # ── Feature 1: Edge density ──
    grad = sobel(gray)
    edge_density = float(grad[mask > 0].mean())

    # ── Feature 2 & 3: Local variance stats ──
    patch = 7
    local_mean = uniform_filter(gray, size=patch)
    local_sq   = uniform_filter(gray ** 2, size=patch)
    local_var  = np.maximum(local_sq - local_mean ** 2, 0)
    local_std  = np.sqrt(local_var)

    std_vals = local_std[mask > 0]
    local_std_mean = float(std_vals.mean())
    std_of_std     = float(std_vals.std())

    return edge_density, local_std_mean, std_of_std


def is_textured_box(mask: np.ndarray, image: np.ndarray,
                    debug: bool = False) -> bool:
    """
    Decide whether a mask region has real-object texture.

    Returns True if:
      - edge_density   >= TEXTURE_MIN_EDGE_DENSITY  (surface detail)
      - local_std_mean >= TEXTURE_MIN_LOCAL_STD     (non-uniformity)
      - std_of_std     <= TEXTURE_MAX_STD_OF_STD    (not bimodal)
        (last check skipped if TEXTURE_MAX_STD_OF_STD == 0)
    """
    ed, ls, sos = compute_texture_features(mask, image)

    detail_ok   = ed >= TEXTURE_MIN_EDGE_DENSITY
    variance_ok = ls >= TEXTURE_MIN_LOCAL_STD
    bimodal_ok  = (TEXTURE_MAX_STD_OF_STD == 0
                   or sos <= TEXTURE_MAX_STD_OF_STD)

    if debug:
        tag = "PASS" if (detail_ok and variance_ok and bimodal_ok) else "REJECT"
        print(f"  [texture] ed={ed:.4f} ls={ls:.4f} sos={sos:.4f} → {tag}")

    return detail_ok and variance_ok and bimodal_ok


# ──────────────────────────────────────────────────────────────
# Morphology & I/O
# ──────────────────────────────────────────────────────────────

def morphological_cleanup(mask, open_r=2, close_r=3):
    if open_r > 0:
        mask = opening(mask, disk(open_r))
    if close_r > 0:
        mask = closing(mask, disk(close_r))
    return mask


def save_box_rgba(image: np.ndarray, mask: np.ndarray, out_path: str):
    """Save an RGBA image where the box is opaque and background is transparent."""
    h, w = image.shape[:2]
    rgba = np.zeros((h, w, 4), dtype=np.uint8)
    rgba[:, :, :3] = image
    rgba[:, :, 3] = (mask.astype(np.uint8)) * 255
    Image.fromarray(rgba, "RGBA").save(out_path)


def save_overlay(image: np.ndarray, boxes_info, out_path: str):
    """
    Save a quick composite showing all accepted boxes with bounding boxes.
    Uses PIL only (no matplotlib) for speed.
    """
    from PIL import ImageDraw
    pil_img = Image.fromarray(image)
    draw = ImageDraw.Draw(pil_img)
    for idx, (_, bbox, _) in enumerate(boxes_info):
        x1, y1, x2, y2 = bbox
        draw.rectangle([x1, y1, x2, y2], outline="#00FF00", width=3)
        draw.text((x1 + 4, y1 + 4), str(idx), fill="#00FF00")
    pil_img.save(out_path)

print("Helper functions defined.")

## 3. Load SAM2 Models

In [ ]:
# Build shared backbone
sam2 = build_sam2(CONFIG, CHECKPOINT, device=device)

# Stage 1: Automatic mask generator
mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2,
    points_per_side=POINTS_PER_SIDE,
    pred_iou_thresh=PRED_IOU_THRESH,
    stability_score_thresh=STABILITY_SCORE_THRESH,
    crop_n_layers=CROP_N_LAYERS,
    crop_n_points_downscale_factor=CROP_N_POINTS_DOWNSCALE_FACTOR,
    min_mask_region_area=MIN_MASK_REGION_AREA,
    output_mode="binary_mask",
)

# Stage 4: Predictor for bbox-prompted refinement
predictor = SAM2ImagePredictor(sam_model=sam2)

print("SAM2 models loaded.")
print(f"  - Mask generator: points_per_side={POINTS_PER_SIDE}")
print(f"  - Predictor: ready for bbox-prompted refinement")

## 4. Main Processing Loop

Per-image pipeline:
1. Generate candidate masks with automatic generator.
2. Filter by geometry (box-like) + frame boundary (fully in view).
3. Filter by texture *(if `ENABLE_TEXTURE_FILTER`)* — kills brick-wall false positives.
4. Refine each candidate with `SAM2ImagePredictor` + bbox prompt.
5. Morphological clean-up.
6. Save each valid box as a separate RGBA image + one overlay diagnostic.

In [ ]:
image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')
image_files = sorted([
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith(image_extensions)
])

if not image_files:
    print(f"No images found in {INPUT_DIR}")
else:
    print(f"Found {len(image_files)} image(s).\n")

for img_file in image_files:
    t0 = time.time()
    img_path = os.path.join(INPUT_DIR, img_file)
    base_name = os.path.splitext(img_file)[0]
    out_subdir = os.path.join(OUTPUT_DIR, base_name)
    os.makedirs(out_subdir, exist_ok=True)

    # --------------------------------------------------------------
    # Load and resize image (prevent OOM)
    # --------------------------------------------------------------
    pil_img = Image.open(img_path).convert("RGB")
    if max(pil_img.size) > MAX_IMAGE_SIZE:
        pil_img.thumbnail((MAX_IMAGE_SIZE, MAX_IMAGE_SIZE), Image.Resampling.LANCZOS)
    image = np.array(pil_img)
    img_h, img_w = image.shape[:2]
    print(f"[{base_name}] {img_w}x{img_h} — loading (resized)...")

    # --------------------------------------------------------------
    # Stage 1: Generate candidate masks
    # --------------------------------------------------------------
    torch.cuda.empty_cache()
    gc.collect()

    with torch.no_grad():
        candidates = mask_generator.generate(image)
    print(f"  Stage 1: {len(candidates)} candidate masks generated.")

    # --------------------------------------------------------------
    # Stage 2: Geometric + boundary filtering
    # --------------------------------------------------------------
    passed = []
    for m in candidates:
        mask = m["segmentation"]
        bbox = get_mask_bbox(mask)
        if bbox is None:
            continue
        if not is_fully_in_frame(bbox, img_h, img_w, margin=BOUNDARY_MARGIN):
            continue
        if not is_box_like(mask, img_h, img_w):
            continue
        passed.append((mask, bbox))
    print(f"  Stage 2: {len(passed)} masks passed geometry + frame filter.")

    # ──────────────────────────────────────────────────────────────
    # Stage 2.5: Texture-based false-positive filter
    # ──────────────────────────────────────────────────────────────
    if ENABLE_TEXTURE_FILTER and passed:
        texture_passed = []
        for mask, bbox in passed:
            if is_textured_box(mask, image):
                texture_passed.append((mask, bbox))
        n_filtered = len(passed) - len(texture_passed)
        print(f"  Stage 2.5 texture: {len(texture_passed)} passed, "
              f"{n_filtered} removed (brick-wall / low-detail).")
        passed = texture_passed
    else:
        print(f"  Stage 2.5 texture: skipped (ENABLE_TEXTURE_FILTER=False).")

    # --------------------------------------------------------------
    # Stage 3: Refine with predictor (bbox prompt)
    # --------------------------------------------------------------
    refined = []
    if REFINE_WITH_PREDICTOR and passed:
        predictor.set_image(image)
        for mask, bbox in passed:
            input_box = np.array(bbox)  # [x1, y1, x2, y2]
            with torch.no_grad():
                masks_pred, scores_pred, _ = predictor.predict(
                    point_coords=None,
                    point_labels=None,
                    box=input_box[None, :],
                    multimask_output=True,
                )
            best_idx = int(np.argmax(scores_pred))
            best_mask = masks_pred[best_idx]
            best_score = float(scores_pred[best_idx])
            if best_score >= REFINE_MIN_SCORE:
                refined.append((best_mask, bbox, best_score))
        print(f"  Stage 3: {len(refined)} masks after predictor refinement.")
    else:
        refined = [(mask, bbox, 1.0) for mask, bbox in passed]
        print(f"  Stage 3: skipped (using Stage-2 masks directly).")

    # --------------------------------------------------------------
    # Stage 4: Post-process + Save
    # --------------------------------------------------------------
    if not refined:
        print(f"  No valid boxes found — nothing saved.\n")
        del candidates, passed, image
        torch.cuda.empty_cache()
        gc.collect()
        continue

    for i, (mask, bbox, score) in enumerate(refined):
        mask = morphological_cleanup(
            mask,
            open_r=MORPH_OPEN_RADIUS,
            close_r=MORPH_CLOSE_RADIUS,
        )
        out_path = os.path.join(out_subdir, f"box_{i:03d}_score{score:.2f}.png")
        save_box_rgba(image, mask, out_path)

    overlay_path = os.path.join(out_subdir, "_overlay_boxes.png")
    save_overlay(image, refined, overlay_path)

    elapsed = time.time() - t0
    print(f"  Saved {len(refined)} box(es) to {out_subdir}/ ({elapsed:.1f}s)\n")

    # Clean up per-image memory
    del candidates, passed, refined, image
    torch.cuda.empty_cache()
    gc.collect()

print("\n=== All images processed ===")
print(f"Output directory: {OUTPUT_DIR}")

## 5. Optional — YOLO-based Box Detection

If you have a trained box detector (e.g. YOLOv8), skip Stage 1–2 and feed bboxes straight to Stage 3.
This bypasses both the geometry and texture filters (YOLO handles class-specific detection).

In [ ]:
# ------------------------------------------------------------------
# YOLO Alternative — uncomment and modify if you have a detector
# ------------------------------------------------------------------
# from ultralytics import YOLO
# yolo = YOLO("/path/to/your/box_detector.pt")
#
# for img_file in image_files:
#     img_path = os.path.join(INPUT_DIR, img_file)
#     image = np.array(Image.open(img_path).convert("RGB"))
#     img_h, img_w = image.shape[:2]
#     base_name = os.path.splitext(img_file)[0]
#     out_subdir = os.path.join(OUTPUT_DIR, base_name)
#     os.makedirs(out_subdir, exist_ok=True)
#
#     results = yolo(img_path, verbose=False)[0]
#     bboxes = []
#     for box in results.boxes:
#         x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
#         conf = float(box.conf[0])
#         if conf < 0.5:
#             continue
#         if not is_fully_in_frame((x1, y1, x2, y2), img_h, img_w, margin=BOUNDARY_MARGIN):
#             continue
#         bboxes.append((x1, y1, x2, y2))
#
#     predictor.set_image(image)
#     for i, bbox in enumerate(bboxes):
#         input_box = np.array(bbox)
#         with torch.no_grad():
#             masks_pred, scores_pred, _ = predictor.predict(
#                 point_coords=None,
#                 point_labels=None,
#                 box=input_box[None, :],
#                 multimask_output=True,
#             )
#         best_idx = int(np.argmax(scores_pred))
#         best_mask = masks_pred[best_idx]
#         best_mask = morphological_cleanup(best_mask, open_r=2, close_r=3)
#         save_box_rgba(image, best_mask, os.path.join(out_subdir, f"box_{i:03d}.png"))
#     print(f"{base_name}: {len(bboxes)} boxes via YOLO → SAM2")

print("YOLO alternative cell ready (currently commented out).")

## 6. Tuning Guide

### General

| Symptom | Fix | Parameter |
|---|---|---|
| Missing small boxes | Raise point density | `POINTS_PER_SIDE = 32` |
| Too many false positives (non-box objects) | Raise rectangularity | `MIN_RECTANGULARITY = 0.65` |
| Broken boxes rejected | Lower rectangularity | `MIN_RECTANGULARITY = 0.35` |
| Edge boxes still segmented | Increase margin | `BOUNDARY_MARGIN = 15` |
| Masks have holes / noise | Adjust morphology | `MORPH_CLOSE_RADIUS = 5` |
| Processing too slow | Lower points + skip refinement | `POINTS_PER_SIDE = 16`, `REFINE_WITH_PREDICTOR = False` |
| Masks bleed into background | Enable predictor refinement | `REFINE_WITH_PREDICTOR = True` |
| Need real-time speed | Use YOLO detector + SAM2 predictor only | See Section 5 |

### Texture Filter

| Symptom | Fix | Parameter |
|---|---|---|
| Brick walls still pass | Raise edge density threshold | `TEXTURE_MIN_EDGE_DENSITY = 0.06` |
| Brick walls still pass | Raise local std threshold | `TEXTURE_MIN_LOCAL_STD = 0.025` |
| Brick walls still pass | Lower std-of-std ceiling | `TEXTURE_MAX_STD_OF_STD = 0.02` |
| Real boxes incorrectly removed | Lower edge density | `TEXTURE_MIN_EDGE_DENSITY = 0.02` |
| Real boxes incorrectly removed | Lower local std | `TEXTURE_MIN_LOCAL_STD = 0.01` |
| Real boxes incorrectly removed | Disable bimodal check | `TEXTURE_MAX_STD_OF_STD = 0` |
| Want to see per-mask feature values | Enable debug | `is_textured_box(mask, image, debug=True)` |
| Texture filter too aggressive overall | Disable it | `ENABLE_TEXTURE_FILTER = False` |